In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv('loan_approval_dataset.csv')

df.head()

,Age,Salary,Credit_Score,Loan_Amount,Loan_Term,Employment_Status,Residence_Type,Previous_Default,Loan_Approved
0,56,136748,584,38209,36 months,Employed,Owned,Yes,Yes
1,46,25287,815,27424,24 months,Self-Employed,Rented,No,Yes
2,32,146593,398,42396,12 months,Unemployed,Rented,Yes,Yes
3,60,54387,696,11370,24 months,Unemployed,Owned,No,No
4,25,28512,788,14528,12 months,Employed,Owned,No,No


In [ ]:
df.shape


(1000, 9)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Age                1000 non-null   int64 
 1   Salary             1000 non-null   int64 
 2   Credit_Score       1000 non-null   int64 
 3   Loan_Amount        1000 non-null   int64 
 4   Loan_Term          1000 non-null   object
 5   Employment_Status  1000 non-null   object
 6   Residence_Type     1000 non-null   object
 7   Previous_Default   1000 non-null   object
 8   Loan_Approved      1000 non-null   object
dtypes: int64(4), object(5)
memory usage: 70.4+ KB


In [ ]:
df.isnull().sum()

Age                  0
Salary               0
Credit_Score         0
Loan_Amount          0
Loan_Term            0
Employment_Status    0
Residence_Type       0
Previous_Default     0
Loan_Approved        0
dtype: int64

In [ ]:
x = df.drop(columns='Loan_Approved')

y = df['Loan_Approved']

In [ ]:
cat_cols = x.select_dtypes(include='object').columns

num_cols = x.select_dtypes(include='number').columns

In [ ]:
cat_cols

Index(['Loan_Term', 'Employment_Status', 'Residence_Type', 'Previous_Default'], dtype='object')

In [ ]:
num_cols

Index(['Age', 'Salary', 'Credit_Score', 'Loan_Amount'], dtype='object')

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(
    x, y, train_size=0.8, random_state=42
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'),
         ['Employment_Status', 'Residence_Type', 'Previous_Default']),

        ('ordinal', OrdinalEncoder(categories=[
            ['12 months', '24 months', '36 months', '48 months']
        ]), ['Loan_Term'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

In [ ]:
pipeline.fit(xtrain, ytrain)

c:\Users\afree\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Employment_Status',
                                                   'Residence_Type',
                                                   'Previous_Default']),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['12 '
                                                                              'months',
                                                                              '24 '
                                                                              'months',
                                                                              '36 '
                                                                              'months',
                                                                              '48 '
                                                                              'months']]),
                                                  ['Loan_Term'])])),
                ('model', DecisionTreeClassifier(random_state=42))])

In [ ]:
ypred = pipeline.predict(xtest)
xpred = pipeline.predict(xtrain)

print(ypred)

In [ ]:
print(accuracy_score(ytrain, xpred))

1.0


In [ ]:
print("Classification Report:")
print(classification_report(ytrain, xpred))

Classification Report:
              precision    recall  f1-score   support

          No       1.00      1.00      1.00       397
         Yes       1.00      1.00      1.00       403

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800



In [ ]:
print("Accuracy:", accuracy_score(ytest, ypred))

print("Confusion Matrix:")
print(confusion_matrix(ytest, ypred))

print("Classification Report:")
print(classification_report(ytest, ypred))

Accuracy: 0.525
Confusion Matrix:
[[45 50]
 [45 60]]
Classification Report:
              precision    recall  f1-score   support

          No       0.50      0.47      0.49        95
         Yes       0.55      0.57      0.56       105

    accuracy                           0.53       200
   macro avg       0.52      0.52      0.52       200
weighted avg       0.52      0.53      0.52       200



In [ ]:
result = pd.DataFrame({
    'Actual': ytest.values,
    'Predicted': ypred
})

result.head(10)

,Actual,Predicted
0,No,No
1,Yes,No
2,No,No
3,Yes,Yes
4,Yes,No
5,Yes,Yes
6,Yes,No
7,Yes,Yes
8,No,Yes
9,No,No


In [ ]:
new_data = pd.DataFrame({
    'Age': [25],
    'Salary': [50000],
    'Credit_Score': [700],
    'Loan_Amount': [20000],
    'Loan_Term': ['24 months'],
    'Employment_Status': ['Employed'],
    'Residence_Type': ['Rented'],
    'Previous_Default': ['No']
})

prediction = pipeline.predict(new_data)

print("Loan Approval:", prediction[0])

Loan Approval: No
